# 03 — Item text embeddings

The technique I'm borrowing here: embed each item's text with a language
model, and feed that embedding into the item tower as a **frozen** input —
the two-tower model never updates these weights, it only learns how to use
them. Two reasons that matters:

1. It lets the model use semantic similarity between items it has almost no
   interaction data for ("this movie's description reads like that one"),
   which raw ids alone can't express.
2. Freezing them keeps the item tower's trainable parameter count small — the
   model is learning a *projection* of a pretrained semantic space, not
   learning semantics from scratch out of ~100K ratings.

I'm using [`sentence-transformers/all-MiniLM-L6-v2`](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2):
small, CPU-friendly, well-known, no GPU needed for ~9,700 short texts.


In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

DATA_DIR = Path("..") / "data" / "raw" / "ml-latest-small"
PROCESSED_DIR = Path("..") / "data" / "processed"

movies = pd.read_csv(DATA_DIR / "movies.csv")
tags = pd.read_csv(DATA_DIR / "tags.csv")
movies.shape, tags.shape


W1024 13:02:54.244000 52146 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


((9742, 3), (3683, 4))

## Building the text per movie

From notebook 1: only ~16% of movies have any user-submitted tag. So the text
fed to the embedder is tags when they exist, with title + genres **always**
appended as grounding context — not just a fallback for the 84% with no tags,
but because tags alone can be noisy/sparse even when present (one or two
words), and title/genre gives the embedding something stable to anchor on.


In [2]:
tags_by_movie = (
    tags.groupby("movieId")["tag"]
    .apply(lambda s: ", ".join(sorted(set(s.str.lower()))))
)

def genres_to_text(genres: str) -> str:
    if genres == "(no genres listed)":
        return ""
    return ", ".join(genres.split("|"))


def build_text(row) -> str:
    tag_text = tags_by_movie.get(row["movieId"], "")
    genre_text = genres_to_text(row["genres"])
    parts = [row["title"]]
    if genre_text:
        parts.append(f"genres: {genre_text}")
    if tag_text:
        parts.append(f"tags: {tag_text}")
    return ". ".join(parts)


movies["text"] = movies.apply(build_text, axis=1)
movies[["movieId", "text"]].sample(5, random_state=1)


,movieId,text
6514,53894,"Sicko (2007). genres: Documentary, Drama"
619,784,"Cable Guy, The (1996). genres: Comedy, Thriller"
3818,5348,"Hollywood Ending (2002). genres: Comedy, Drama"
3530,4825,"Swamp, The (Ciénaga, La) (2001). genres: Comed..."
1225,1626,"Fire Down Below (1997). genres: Action, Drama,..."


In [3]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embeddings = model.encode(
    movies["text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)
embeddings.shape


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/153 [00:00<?, ?it/s]

(9742, 384)

`normalize_embeddings=True` means these are unit vectors — so a dot product
between two of them is a cosine similarity. Worth checking that actually
produces sensible neighbors before trusting it as a model input.


In [4]:
def nearest(movie_title_substr: str, k: int = 6):
    idx = movies.index[movies["title"].str.contains(movie_title_substr, case=False)][0]
    sims = embeddings @ embeddings[idx]
    top = np.argsort(-sims)[: k + 1]
    return movies.iloc[top][["title", "genres"]].assign(similarity=sims[top])


nearest("Toy Story")


,title,genres,similarity
0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1.000000
2355,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,0.771514
7355,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX,0.760687
1757,"Bug's Life, A (1998)",Adventure|Animation|Children|Comedy,0.747151
205,"Goofy Movie, A (1995)",Animation|Children|Comedy|Romance,0.739643
1674,Toys (1992),Comedy|Fantasy,0.702280
3595,"Toy, The (1982)",Comedy,0.684763


## Save for notebook 4

These are an intermediate artifact (`data/processed/`, regenerable from the
raw data + this notebook, so not committed to `artifacts/`) — the item tower
in notebook 4 consumes `item_text_embeddings.npz`, and what actually gets
committed for serving is the item tower's *output* (a much smaller, trained
projection), not this raw 384-dim text embedding.


In [5]:
movie_ids = movies["movieId"].to_numpy()
np.savez(
    PROCESSED_DIR / "item_text_embeddings.npz",
    movie_ids=movie_ids,
    embeddings=embeddings.astype("float32"),
)
print(f"saved {embeddings.shape[0]} item text embeddings, dim={embeddings.shape[1]}")


saved 9742 item text embeddings, dim=384


## My take

Not an ADR by itself (no real architectural fork here — "embed the text" was
the ask, not a decision with a plausible alternative I'm weighing), but worth
noting for ADR-004 later: these text embeddings are exactly the kind of
feature that should matter *most* for cold-start items (new movies with no
interaction history yet, only metadata) — which this dataset doesn't really
let me test (`ml-latest-small` doesn't have item-cold-start movies added after
the dataset snapshot). Keep that gap in mind when reading the cold-start
ablation in notebook 4 — it's measuring *user* cold-start, not item cold-start.
